In [ ]:
import numpy as np

depth = np.load("/Users/adeleyounis/Downloads/depth_registered_block.npy")  # depth in meters usually
print(depth.shape)


In [ ]:
import cv2
import numpy as np
import open3d as o3d

fx= 596.25827383
fy=593.35350108
cx= 328.00224565
cy= 246.72323964
rgb = cv2.imread("/Users/adeleyounis/Downloads/block.png")
hsv = cv2.cvtColor(rgb, cv2.COLOR_BGR2HSV)

# --- ROI bounding box (adjust if needed) ---
y1, y2 = 270, 320
x1, x2 = 380, 460

# blank mask full image size
mask = np.zeros((rgb.shape[0], rgb.shape[1]), dtype=np.uint8)

# threshold only inside ROI
hsv_roi = hsv[y1:y2, x1:x2]

lower = np.array([10, 70, 70])
upper = np.array([30, 255, 255])

mask_roi = cv2.inRange(hsv_roi, lower, upper)

# put ROI mask back into the full mask
mask[y1:y2, x1:x2] = mask_roi

# # optional blur + morphology to clean noise
mask = cv2.medianBlur(mask, 5)
# mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5,5), np.uint8))
# mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((7,7), np.uint8))

# show result
# cv2.imshow("Filtered Block Mask", mask)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

ys, xs = np.where(mask)
zs = depth[ys, xs]

# Convert to point cloud
Xs = (xs - cx) * zs / fx
Ys = (ys - cy) * zs / fy
Zs = zs

# Dimensions
width  = Xs.max() - Xs.min()
length = Ys.max() - Ys.min()

points = np.vstack((Xs, Ys, Zs)).T
colors = rgb[ys, xs] / 255.0

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(colors)

o3d.io.write_point_cloud("block_only.ply", pcd)
print("Saved block_only.ply")


print(width)
print(length)